# The pipeline, step by step

This notebook runs each step of the ingest pipeline on the golden filing (Maroc Telecom, Rapport
financier annuel 2025) with the same functions the app uses, and shows what each step produces.
New steps are tried here first, then moved into `bloomberg_invest`.

| Step | Code | Status |
| --- | --- | --- |
| 1. Parse | read from the app: `bloomberg ingest` | in the app (#20) |
| 2. Read blocks | `DoclingParser.read_blocks` | in the app (#21 PR A) |
| 3. Chunk | `domain/chunking.py` | in the app (#21 PR A) |
| 4. Embed | sentence-transformers, below | **prototype** for #21 PR B |
| 5. Search | full-text, vector and hybrid, below | **prototype** for #21 PR B |

It reads the app's own parse of the filing, so it shows exactly what the app holds. Start the
stack and ingest the golden PDF first (README), then run it from `backend/`:

```bash
uv run --env-file ../.env --extra ingest --group notebook jupyter lab notebooks/pipeline.ipynb
```

Outputs are stripped before commit (`nbstripout`): they show pages of the AMMC report, which is
not redistributed. It only reads from Neo4j.

In [ ]:
import hashlib
import math
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypdfium2 as pdfium
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from neo4j import GraphDatabase, NotificationDisabledClassification

from bloomberg_invest.adapters.parsing.docling_parser import DoclingParser
from bloomberg_invest.adapters.store.neo4j_store import Neo4jStore
from bloomberg_invest.adapters.tokens import TiktokenCounter
from bloomberg_invest.domain.chunking import SectionRef, chunk
from bloomberg_invest.settings import Settings

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().parent
DATA = ROOT / "data"
PDF = DATA / "inbox" / "Maroc_Telecom_RFA_2025.pdf"
WORK = DATA / "notebook"  # gitignored, like all of data/
WORK.mkdir(parents=True, exist_ok=True)
CAP = 2000  # CHUNK_CAP

pd.set_option("display.max_colwidth", 90)
assert PDF.exists(), f"put the golden PDF at {PDF} (golden.md gives the link and SHA-256)"
print(PDF, hashlib.sha256(PDF.read_bytes()).hexdigest()[:12])

## 1. Parse

The app parses each filing once, with Docling in its Docker image, and saves Docling's output with
the parse Run. This step finds the golden filing in the app's graph by its SHA-256 and reads that
Run, so every step below starts from the app's own parse. Parsing again on another machine can
cut headings and paragraphs differently (#21).

In [ ]:
settings = Settings.from_env()
sha256 = hashlib.sha256(PDF.read_bytes()).hexdigest()
with GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password),
    notifications_disabled_classifications=[NotificationDisabledClassification.UNRECOGNIZED],
) as driver:
    store = Neo4jStore(driver)
    details = (store.get_document(d.id) for d in store.list_documents())
    detail = next((d for d in details if d.sha256 == sha256), None)
assert detail, "ingest the golden PDF into the app first (README: docker compose run --rm ingest)"
assert detail.run.output, "the parse run saved no output; re-ingest"

parser = DoclingParser()
output = DATA / detail.run.output
run = detail.run
print(f"document {detail.id}, {detail.page_count} pages")
print(
    f"parse run {run.id}: {run.tool} {run.tool_version} {run.config}, finished {run.finished_at:%Y-%m-%d %H:%M}"
)
print(output.relative_to(DATA))

### The section outline

In [ ]:
def walk(nodes, depth=0):
    for n in nodes:
        yield depth, n
        yield from walk(n.children, depth + 1)


outline = list(walk(detail.sections))
print(
    f"{len(outline)} sections; by level: {dict(sorted(Counter(n.level for _, n in outline).items()))}"
)
for depth, n in outline[:45]:
    print(f"{'    ' * depth}{n.title[:70]}  (p. {n.page})")

### A page with its headings

Each Section keeps its heading's box in PDF points from the top-left. The viewer draws highlights
the same way.

In [ ]:
pdf = pdfium.PdfDocument(PDF)
sizes = {p.number: (p.width, p.height) for p in detail.pages}


def show_page(number, boxes=(), title=None, scale=1.4):
    """Draw a page with boxes: (left, top, right, bottom, colour, label)."""
    image = pdf[number - 1].render(scale=scale).to_pil()
    _, ax = plt.subplots(figsize=(9, 9 * image.height / image.width))
    ax.imshow(image)
    for left, top, right, bottom, colour, label in boxes:
        ax.add_patch(
            Rectangle(
                (left * scale, top * scale),
                (right - left) * scale,
                (bottom - top) * scale,
                fill=False,
                lw=1.6,
                ec=colour,
            )
        )
        if label:  # in the right margin, clear of the page's own text
            ax.text(right * scale + 4, top * scale + 8, label, color=colour, fontsize=7)
    ax.set_title(title or f"Page {number}")
    ax.axis("off")
    plt.show()


PAGE = 29
show_page(PAGE, [(*n.bbox, "crimson", None) for _, n in outline if n.page == PAGE and n.bbox])

## 2. Read blocks

The chunker reads the saved Docling output back as headings, paragraphs and tables in reading
order. A paragraph keeps every box it covers; `numbered` marks list items and numbered points.

In [ ]:
blocks = parser.read_blocks(output)
frame = pd.DataFrame(
    [
        {
            "kind": b.kind,
            "page": b.page,
            "pages": len(b.pages),
            "numbered": b.numbered,
            "columns": b.columns,
            "rows": len(b.rows),
            "text": (b.text or b.header)[:90],
        }
        for b in blocks
    ]
)
print(frame.kind.value_counts().to_dict())
frame.head(25)

## 3. Chunk

The rules (#19 R2): a table is one Chunk, joined when it continues on the next page; a section's
paragraphs form one Chunk under the cap; a longer section splits at numbered points, otherwise
evenly; a paragraph is never split. Token counts use `cl100k_base`.

In [ ]:
count = TiktokenCounter()
refs = [SectionRef(id=n.id, title=n.title, page=n.page) for _, n in outline]
titles = {r.id: r.title for r in refs}
chunks = chunk(blocks, refs, CAP, count)

table = pd.DataFrame(
    [
        {
            "index": c.index,
            "kind": c.kind,
            "section_id": c.section_id,
            "section": titles.get(c.section_id, "")[:45],
            "pages": f"{c.pages[0]}-{c.pages[-1]}" if len(c.pages) > 1 else str(c.pages[0]),
            "tokens": c.token_count,
            "paragraphs": len(c.paragraphs),
            "cites": ", ".join(titles[s][:12] for s in c.refers_to),
        }
        for c in chunks
    ]
)
print(
    f"{len(chunks)} chunks ({(table.kind == 'table').sum()} tables), "
    f"{sum(len(c.paragraphs) for c in chunks)} paragraphs"
)
table.drop(columns="section_id").head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, kind in zip(axes, ["text", "table"]):
    ax.hist(table[table.kind == kind].tokens, bins=40, color="#4c72b0")
    ax.axvline(CAP, color="crimson", ls="--", label=f"cap {CAP}")
    ax.set_title(f"{kind} chunks: tokens")
    ax.legend()
plt.show()
print("text chunks over the cap (single paragraphs only):")
table[(table.kind == "text") & (table.tokens > CAP)]

### Sections split by the cap, and tables joined across pages

golden.md lists the true multi-page tables (§5): the key audit matters on pp. 74-75 and 86-87.

In [ ]:
# Group by Section id: titles repeat ("État de résultat global" is several Sections).
text_chunks = table[table.kind == "text"]
split = text_chunks.groupby("section_id").filter(lambda g: len(g) > 1)
if split.empty:
    print(f"No section's text is over the cap of {CAP} tokens in this parse.")
else:
    display(
        split.groupby(["section_id", "section"]).agg(
            chunks=("index", "count"), tokens=("tokens", "sum")
        )
    )
joined = table[(table.kind == "table") & table.pages.str.contains("-")]
joined

In [ ]:
c = chunks[int(joined.iloc[0]["index"])] if len(joined) else None
display(Markdown(c.text[:3000] if c else "No table was joined."))

### The paragraphs of one page, coloured by Chunk

In [ ]:
PAGE = 10
colours = plt.cm.tab10.colors
on_page = [(c, p) for c in chunks for p in c.paragraphs if PAGE in p.pages]
order = {c.index: i for i, c in enumerate(dict.fromkeys(c for c, _ in on_page))}
boxes, labelled = [], set()
for c, p in on_page:
    for pg, left, top, right, bottom in p.boxes:
        if pg == PAGE:
            label = None if c.index in labelled else f"chunk {c.index}"
            labelled.add(c.index)
            boxes.append((left, top, right, bottom, colours[order[c.index] % 10], label))
show_page(PAGE, boxes)

### Notes a Chunk cites

`REFERS_TO` links a Chunk to the NOTE it cites. golden.md §6 lists p. 29's "(cf. note 14)",
"(cf. note 3)", "(cf. note 8)" and "la note 2".

In [ ]:
table[(table.cites != "") & table.pages.str.startswith("29")]

## 4. Embed (prototype for #21 PR B)

The search unit is the Paragraph. Two open models from the research, run locally:

| Model | Licence | Dims | Query prefix |
| --- | --- | --- | --- |
| `Qwen/Qwen3-Embedding-0.6B` | Apache 2.0 | 1,024 | `query` prompt |
| `BAAI/bge-m3` | MIT | 1,024 | none |

Vectors are cached in `data/notebook/` by model and paragraph text, so a rerun is instant. Azure
`text-embedding-3-large` joins once its deployment is confirmed.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELS = {
    "qwen3-0.6b": {"name": "Qwen/Qwen3-Embedding-0.6B", "query_prompt": "query"},
    "bge-m3": {"name": "BAAI/bge-m3", "query_prompt": None},
}
paragraphs = [(c, p) for c in chunks for p in c.paragraphs]
texts = [p.text for _, p in paragraphs]
key = hashlib.sha256("\n\0".join(texts).encode()).hexdigest()[:12]
loaded, vectors, timings = {}, {}, {}

for label, spec in MODELS.items():
    model = loaded[label] = SentenceTransformer(spec["name"])
    model.max_seq_length = (
        1024  # long tables are cut for the vector; their full text stays searchable
    )
    cache = WORK / f"embeddings-{label}-{key}.npy"
    started = time.perf_counter()
    if cache.exists():
        vectors[label] = np.load(cache)
    else:
        vectors[label] = model.encode(
            texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True
        )
        np.save(cache, vectors[label])
    timings[label] = time.perf_counter() - started
    print(f"{label}: {vectors[label].shape} in {timings[label]:.0f}s on {model.device}")

## 5. Search (prototype for #21 PR B)

- **Full-text:** BM25 over the paragraphs, lower-cased with accents folded: the stand-in for
  Neo4j's `standard-folding` index.
- **Vector:** cosine similarity with each model.
- **Hybrid:** reciprocal rank fusion of the two, as `HybridCypherRetriever` combines them.

In [ ]:
def fold(text):
    text = unicodedata.normalize("NFKD", text.lower())
    return re.findall(r"\w+", "".join(ch for ch in text if not unicodedata.combining(ch)))


docs = [fold(t) for t in texts]
df = Counter(term for d in docs for term in set(d))
avg = sum(map(len, docs)) / len(docs)


def bm25(query, k1=1.2, b=0.75):
    terms = fold(query)
    scores = np.zeros(len(docs))
    for i, d in enumerate(docs):
        tf = Counter(d)
        for t in terms:
            if tf[t]:
                idf = math.log(1 + (len(docs) - df[t] + 0.5) / (df[t] + 0.5))
                scores[i] += idf * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avg))
    return scores


def vector(query, label):
    spec, model = MODELS[label], loaded[label]
    kwargs = {"prompt_name": spec["query_prompt"]} if spec["query_prompt"] else {}
    q = model.encode([query], normalize_embeddings=True, **kwargs)[0]
    return vectors[label] @ q


def ranks(scores):
    return np.argsort(-scores)


def hybrid(query, label, k=60):
    fused = np.zeros(len(texts))
    for scores in (bm25(query), vector(query, label)):
        for rank, i in enumerate(ranks(scores)):
            fused[i] += 1 / (k + rank + 1)
    return fused


def top(scores, n=5):
    return [(i, paragraphs[i][1].page, texts[i][:110]) for i in ranks(scores)[:n]]

### AC3: "dividende proposé" must return the p. 10 dividend paragraph in the top 5

In [ ]:
QUERY = "dividende proposé"
for name, scores in (
    [("full-text", bm25(QUERY))]
    + [(f"vector {m}", vector(QUERY, m)) for m in MODELS]
    + [(f"hybrid {m}", hybrid(QUERY, m)) for m in MODELS]
):
    print(f"\n{name}")
    for i, page, text in top(scores):
        print(f"  p. {page:>2}  {text}")

In [ ]:
best = ranks(hybrid(QUERY, "qwen3-0.6b"))[0]
c, p = paragraphs[best]
show_page(
    p.page,
    [(l, t, r, b, "orange", "top hit") for (pg, l, t, r, b) in p.boxes if pg == p.page],
    title=f'"{QUERY}": top hit, p. {p.page}',
)

### The golden questions

golden.md §8 gives each question's cited pages. A hit is any top-5 Paragraph on a cited page.
The questions are in English and the report in French, so the vector side must cross languages.

In [ ]:
QUESTIONS = {
    "Q1": ("What does Maroc Telecom forecast for 2026?", {10}),
    "Q2": (
        (
            "Which market risks does the group describe, and what is the impact of a 1% "
            "depreciation of the dirham against the euro?"
        ),
        {68, 69, 70},
    ),
    "Q3": (
        "What happened with the Wana Corporate dispute in 2025 and how did it affect results?",
        {68, 10, 15},
    ),
    "Q4": (
        (
            "With which related parties did Maroc Telecom transact in 2025, and how much revenue "
            "came from Etisalat?"
        ),
        {66, 67},
    ),
    "Q5": ("What was consolidated revenue (IFRS) for FY2025?", {24}),
    "Q6": (
        "What was consolidated net debt at 31 Dec 2025 and the net debt / EBITDA ratio?",
        {56, 9},
    ),
    "Q7": ("What was FY2025 net income attributable to the group?", {24, 9, 10}),
    "Q8": ("What was Maroc Telecom's company-only (CGNC) net income for 2025?", {81, 85}),
    "Q9": ("Who are Maroc Telecom's shareholders at 31 Dec 2025?", {27}),
    "Q10": (
        "Which fully consolidated subsidiaries are not wholly owned, and what is the interest?",
        {40, 63},
    ),
    "AC3": ("dividende proposé", {10}),
}


def hit(scores, pages, n=5):
    return any(paragraphs[i][1].page in pages for i in ranks(scores)[:n])


rows = []
for q, (question, pages) in QUESTIONS.items():
    row = {"question": q, "full-text": hit(bm25(question), pages)}
    for m in MODELS:
        row[f"vector {m}"] = hit(vector(question, m), pages)
        row[f"hybrid {m}"] = hit(hybrid(question, m), pages)
    rows.append(row)
scores = pd.DataFrame(rows).set_index("question")
display(scores.replace({True: "hit", False: "-"}))
summary = scores.mean().rename("hit@5").to_frame()
summary["embed seconds"] = [None] + [timings[m] for m in MODELS for _ in (0, 1)]
summary

## What this tells PR B

Read the scorecard above: the model with the best hybrid hit@5 on the golden questions is the
default `EMBEDDER` candidate, weighed against its embedding time. Before choosing, Azure
`text-embedding-3-large` joins the same table. The full-text side here is a BM25 stand-in; the
app uses Neo4j's `standard-folding` index.